# *T.Kanishka cs24b2046*

In [1]:

import cv2
import numpy as np

image = cv2.imread("input.jpg")

if image is None:
    raise FileNotFoundError("Cannot find input.jpg")

points = []

def select_points(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 4:
        points.append([x, y])
        print(f"Selected point: ({x}, {y})")

cv2.namedWindow("Select 4 Corners")
cv2.setMouseCallback("Select 4 Corners", select_points)

while True:
    display = image.copy()

    for i, p in enumerate(points):
        cv2.circle(display, tuple(p), 7, (0, 0, 255), -1)
        cv2.putText(
            display, str(i + 1),
            (p[0] + 8, p[1] - 8),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7,
            (0, 255, 0), 2
        )

    cv2.imshow("Select 4 Corners", display)
    key = cv2.waitKey(20) & 0xFF

    if key == ord('r'):
        points.clear()
    elif key == ord('q') and len(points) == 4:
        break

cv2.destroyAllWindows()

# Automatically order corners:
# top-left, top-right, bottom-right, bottom-left
pts = np.array(points, dtype=np.float32)
rect = np.zeros((4, 2), dtype=np.float32)

s = pts.sum(axis=1)
diff = np.diff(pts, axis=1).ravel()

rect[0] = pts[np.argmin(s)]      # Top-left
rect[2] = pts[np.argmax(s)]      # Bottom-right
rect[1] = pts[np.argmin(diff)]   # Top-right
rect[3] = pts[np.argmax(diff)]   # Bottom-left

# Calculate output dimensions from the selected quadrilateral
tl, tr, br, bl = rect

width = int(max(
    np.linalg.norm(br - bl),
    np.linalg.norm(tr - tl)
))
height = int(max(
    np.linalg.norm(tr - br),
    np.linalg.norm(tl - bl)
))

if width < 2 or height < 2:
    raise ValueError("Invalid corners selected.")

dst = np.float32([
    [0, 0],
    [width - 1, 0],
    [width - 1, height - 1],
    [0, height - 1]
])

H = cv2.getPerspectiveTransform(rect, dst)

result = cv2.warpPerspective(image, H, (width, height))

cv2.imwrite("rectified.jpg", result)

cv2.imshow("Original Image", image)
cv2.imshow("Rectified Image", result)

print("Saved rectified.jpg")
cv2.waitKey(0)
cv2.destroyAllWindows()


Selected point: (14, 102)
Selected point: (95, 229)
Selected point: (140, 46)
Selected point: (185, 151)
Saved rectified.jpg


In [1]:

import cv2
import numpy as np

# --------------------------------------------------
# STEP 1: Read the input image
# --------------------------------------------------

image = cv2.imread("input.jpg")

if image is None:
    print("Error: Image not found!")
    exit()

# Display the input image
cv2.imshow("Original Image", image)

# --------------------------------------------------
# STEP 2: Select the four corners of the planar surface
# --------------------------------------------------

print("Click the four corners in this order:")
print("Top-left, Top-right, Bottom-right, Bottom-left")

points = []

def select_point(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 4:
        points.append([x, y])
        print(f"Point {len(points)}: ({x}, {y})")

        cv2.circle(image, (x, y), 5, (0, 0, 255), -1)
        cv2.imshow("Original Image", image)

cv2.namedWindow("Original Image")
cv2.setMouseCallback("Original Image", select_point)

while len(points) < 4:
    cv2.waitKey(1)

cv2.destroyAllWindows()

src_points = np.array(points, dtype=np.float64)

# --------------------------------------------------
# STEP 3: Define the destination rectangle
# --------------------------------------------------

width = 600
height = 600

dst_points = np.array([
    [0, 0],
    [width - 1, 0],
    [width - 1, height - 1],
    [0, height - 1]
], dtype=np.float64)

# --------------------------------------------------
# STEP 4: Calculate homography manually
# --------------------------------------------------
# For each correspondence (x,y) -> (u,v):
#
# u = (h11*x + h12*y + h13) /
#     (h31*x + h32*y + 1)
#
# v = (h21*x + h22*y + h23) /
#     (h31*x + h32*y + 1)
#
# We set h33 = 1 and solve for 8 unknowns.

A = []
B = []

for i in range(4):
    x, y = src_points[i]
    u, v = dst_points[i]

    A.append([
        x, y, 1, 0, 0, 0,
        -u * x, -u * y
    ])
    B.append(u)

    A.append([
        0, 0, 0, x, y, 1,
        -v * x, -v * y
    ])
    B.append(v)

A = np.array(A, dtype=np.float64)
B = np.array(B, dtype=np.float64)

# Solve A*h = B using Gaussian elimination.
# This implementation does not use np.linalg.solve()
# or cv2.findHomography().

def gaussian_elimination(A, B):
    n = len(B)

    M = np.zeros((n, n + 1), dtype=np.float64)
    M[:, :n] = A
    M[:, n] = B

    # Forward elimination with partial pivoting
    for i in range(n):
        pivot = i

        for j in range(i + 1, n):
            if abs(M[j, i]) > abs(M[pivot, i]):
                pivot = j

        if abs(M[pivot, i]) < 1e-10:
            raise ValueError(
                "Cannot calculate homography. "
                "Select four distinct, non-collinear corners."
            )

        M[[i, pivot]] = M[[pivot, i]]

        for j in range(i + 1, n):
            factor = M[j, i] / M[i, i]

            for k in range(i, n + 1):
                M[j, k] -= factor * M[i, k]

    # Back substitution
    h = np.zeros(n, dtype=np.float64)

    for i in range(n - 1, -1, -1):
        total = M[i, n]

        for j in range(i + 1, n):
            total -= M[i, j] * h[j]

        h[i] = total / M[i, i]

    return h


h = gaussian_elimination(A, B)

H = np.array([
    [h[0], h[1], h[2]],
    [h[3], h[4], h[5]],
    [h[6], h[7], 1.0]
], dtype=np.float64)

print("\nCalculated Homography Matrix:")
print(H)

# --------------------------------------------------
# STEP 5: Apply the homography transformation
# --------------------------------------------------

rectified = cv2.warpPerspective(
    cv2.imread("input.jpg"),
    H,
    (width, height)
)

# --------------------------------------------------
# STEP 6: Display and save the output
# --------------------------------------------------

cv2.imshow("Rectified Image", rectified)

cv2.imwrite("rectified_output.jpg", rectified)

print("\nRectification completed.")
print("Output saved as rectified_output.jpg")

cv2.waitKey(0)
cv2.destroyAllWindows()


Click the four corners in this order:
Top-left, Top-right, Bottom-right, Bottom-left
Point 1: (4, 96)
Point 2: (140, 38)
Point 3: (188, 150)
Point 4: (92, 232)

Calculated Homography Matrix:
[[ 2.47197533e+00 -1.59951345e+00  1.43665390e+02]
 [ 1.18623693e+00  2.78152109e+00 -2.71770972e+02]
 [-2.01762682e-03 -3.69646439e-05  1.00000000e+00]]

Rectification completed.
Output saved as rectified_output.jpg
